# Segmentacija

Jau i?mokome apie objekt? aptikim?, kuris leid?ia nustatyti objekt? viet? vaizde, prognozuojant j? *ribojan?ias d??utes*. Ta?iau kai kurioms u?duotims mums reikia ne tik ribojan?i? d??u?i?, bet ir tikslesnio objekt? lokalizavimo. ?i u?duotis vadinama **segmentacija**.

Segmentacij? galima laikyti **pikseli? klasifikacija**, kur kiekvienam vaizdo pikseliui turime prognozuoti jo klas? (*fonas* yra viena i? klasi?). Yra du pagrindiniai segmentacijos algoritmai:

* **Semantin? segmentacija** nurodo tik pikselio klas? ir neskiria skirting? tos pa?ios klas?s objekt?.
* **Objekt? segmentacija** (angl. instance segmentation) suskaido klases ? skirtingus objektus.

Pavyzd?iui, objekt? segmentacijoje 10 avi? yra skirtingi objektai, o semantin?je segmentacijoje visos avys yra atvaizduojamos kaip viena klas?.

<img src="images/instance_vs_semantic.jpeg" width="50%">

> Vaizdas i? [?io tinklara??io ?ra?o](https://nirmalamurali.medium.com/image-classification-vs-semantic-segmentation-vs-instance-segmentation-625c33a08d50)

Yra ?vairi? neuronini? architekt?r? segmentacijai, ta?iau jos visos turi t? pa?i? strukt?r?:

* **Koduotojas** i?gauna savybes i? ?vesties vaizdo.
* **Dekoduotojas** transformuoja tas savybes ? **kauk?s vaizd?**, kurio dydis ir kanal? skai?ius atitinka klasi? skai?i?.

<img src="images/segm.png" width="80%">

> Vaizdas i? [?ios publikacijos](https://arxiv.org/pdf/2001.05566.pdf)


## B?tinos s?lygos

Prad?kime nuo reikaling? bibliotek? importavimo ir patikrinkime, ar yra GPU, skirtas mokymui.


In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
from torchvision import transforms
from torch import nn
from torch import optim
from tqdm import tqdm
import numpy as np
import torch.nn.functional as F
from skimage.io import imread
from skimage.transform import resize
import os
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
train_size = 0.9
lr = 1e-3
weight_decay = 1e-6
batch_size = 32
epochs = 30

## Duomen? rinkinys

Naudosime dermoskopijos vaizdus, kuriuose matomi ?mogaus apgamai. ?is duomen? rinkinys apima 200 vaizd?, suskirstyt? ? tris klases: tipi?kas apgamas, netipi?kas apgamas ir melanoma. Visi vaizdai taip pat turi atitinkamas **kaukes**, kurios apibr??ia apgam?.

?emiau pateiktas kodas atsisiun?ia duomen? rinkin? i? originalios vietos ir i?skleid?ia j?. Kad ?is kodas veikt?, jums reik?s ?diegti `unrar` ?rank?. J? galite ?diegti naudodami `sudo apt-get install unrar` Linux sistemoje arba atsisi?sdami komand? eilut?s versij? Windows sistemai [?ia](https://www.rarlab.com/rar_add.htm).


In [ ]:
#!apt-get install rar
!wget https://www.dropbox.com/s/k88qukc20ljnbuo/PH2Dataset.rar
!unrar x -Y PH2Dataset.rar

Dabar apibr??ime kod?, skirt? duomen? rinkiniui ?kelti. Visus vaizdus transformuosime ? 256x256 dyd? ir padalinsime duomen? rinkin? ? mokymo ir testavimo dalis. ?i funkcija gr??ina mokymo ir testavimo duomen? rinkinius, kuriuose yra original?s vaizdai ir kauk?s, apibr??ian?ios apgam?.


In [ ]:
def load_dataset(train_part, root='PH2Dataset'):
    images = []
    masks = []

    for root, dirs, files in os.walk(os.path.join(root, 'PH2 Dataset images')):
        if root.endswith('_Dermoscopic_Image'):
            images.append(imread(os.path.join(root, files[0])))
        if root.endswith('_lesion'):
            masks.append(imread(os.path.join(root, files[0])))

    size = (256, 256)
    images = torch.permute(torch.FloatTensor(np.array([resize(image, size, mode='constant', anti_aliasing=True,) for image in images])), (0, 3, 1, 2))
    masks = torch.FloatTensor(np.array([resize(mask, size, mode='constant', anti_aliasing=False) > 0.5 for mask in masks])).unsqueeze(1)

    indices = np.random.permutation(range(len(images)))
    train_part = int(train_part * len(images))
    train_ind = indices[:train_part]
    test_ind = indices[train_part:]

    train_dataset = (images[train_ind, :, :, :], masks[train_ind, :, :, :])
    test_dataset = (images[test_ind, :, :, :], masks[test_ind, :, :, :])

    return train_dataset, test_dataset

train_dataset, test_dataset = load_dataset(train_size)

Dabar nubrai?ykime kelet? vaizd? i? duomen? rinkinio, kad pamatytume, kaip jie atrodo:


In [ ]:
def plotn(n, data, only_mask=False):
    images, masks = data[0], data[1]
    fig, ax = plt.subplots(1, n)
    fig1, ax1 = plt.subplots(1, n)
    for i, (img, mask) in enumerate(zip(images, masks)):
        if i == n:
            break
        if not only_mask:
            ax[i].imshow(torch.permute(img, (1, 2, 0)))
        else:
            ax[i].imshow(img[0])
        ax1[i].imshow(mask[0])
        ax[i].axis('off')
        ax1[i].axis('off')
    plt.show()

plotn(5, train_dataset)

Mes taip pat reik?s duomen? kaupikli?, kad gal?tume pateikti duomenis ? m?s? neuronin? tinkl?.


In [ ]:
train_dataloader = torch.utils.data.DataLoader(list(zip(train_dataset[0], train_dataset[1])), batch_size=batch_size, shuffle=True)
test_dataloader = torch.utils.data.DataLoader(list(zip(test_dataset[0], test_dataset[1])), batch_size=1, shuffle=False)
dataloaders = (train_dataloader, test_dataloader)

## SegNet

Papras?iausia kodavimo-dekodavimo architekt?ra vadinama **SegNet**. Ji naudoja standartin? CNN su konvoliucijomis ir sujungimais kodavimo dalyje, o dekodavimo dalyje ? dekodavimo CNN, kuris apima konvoliucijas ir padidinimus. Taip pat ji remiasi partij? normalizacija, kad s?kmingai apmokyt? daugiapakop? tinkl?.

<img src="images/segnet.png" width="80%">

> Vaizdas i? ?io straipsnio: Badrinarayanan, V., Kendall, A., & Cipolla, R. (2015). [SegNet: Gili konvoliucin?
kodavimo-dekodavimo architekt?ra vaizd? segmentavimui](https://arxiv.org/pdf/1511.00561.pdf)


In [ ]:
class SegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_conv0 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(3,3), padding=1)
        self.act0 = nn.ReLU()
        self.bn0 = nn.BatchNorm2d(16)
        self.pool0 = nn.MaxPool2d(kernel_size=(2,2))

        self.enc_conv1 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3,3), padding=1)
        self.act1 = nn.ReLU()
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=(2,2))

        self.enc_conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3,3), padding=1)
        self.act2 = nn.ReLU()
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 =  nn.MaxPool2d(kernel_size=(2,2))

        self.enc_conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3,3), padding=1)
        self.act3 = nn.ReLU()
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 =  nn.MaxPool2d(kernel_size=(2,2))

        self.bottleneck_conv = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(3,3), padding=1)
        
        self.upsample0 =  nn.UpsamplingBilinear2d(scale_factor=2)
        self.dec_conv0 = nn.Conv2d(in_channels=256, out_channels=128, kernel_size=(3,3), padding=1)
        self.dec_act0 = nn.ReLU()
        self.dec_bn0 = nn.BatchNorm2d(128)

        self.upsample1 =  nn.UpsamplingBilinear2d(scale_factor=2)
        self.dec_conv1 =  nn.Conv2d(in_channels=128, out_channels=64, kernel_size=(3,3), padding=1)
        self.dec_act1 = nn.ReLU()
        self.dec_bn1 = nn.BatchNorm2d(64)

        self.upsample2 = nn.UpsamplingBilinear2d(scale_factor=2)
        
        self.dec_conv2 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=(3,3), padding=1)
        self.dec_act2 = nn.ReLU()
        self.dec_bn2 = nn.BatchNorm2d(32)

        self.upsample3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.dec_conv3 = nn.Conv2d(in_channels=32, out_channels=1, kernel_size=(1,1))

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        e0 = self.pool0(self.bn0(self.act0(self.enc_conv0(x))))
        e1 = self.pool1(self.bn1(self.act1(self.enc_conv1(e0))))
        e2 = self.pool2(self.bn2(self.act2(self.enc_conv2(e1))))
        e3 = self.pool3(self.bn3(self.act3(self.enc_conv3(e2))))

        b = self.bottleneck_conv(e3)

        d0 = self.dec_bn0(self.dec_act0(self.dec_conv0(self.upsample0(b))))
        d1 = self.dec_bn1(self.dec_act1(self.dec_conv1(self.upsample1(d0))))
        d2 = self.dec_bn2(self.dec_act2(self.dec_conv2(self.upsample2(d1))))
        d3 = self.sigmoid(self.dec_conv3(self.upsample3(d2)))
        return d3

Tur?tume ypa? pamin?ti nuostoli? funkcij?, naudojam? segmentacijai. Klasikiniuose automatinio kodavimo ?renginiuose mums reikia i?matuoti dviej? vaizd? pana?um?, ir tam galime naudoti vidutin? kvadratin? paklaid?. Segmentacijoje kiekvienas ta?kas tikslinio kauk?s vaizdo atspindi klas?s numer? (vieno kar?to kodavimo b?du tre?ioje dimensijoje), tod?l turime naudoti klasifikacijai specifines nuostoli? funkcijas - kry?min?s entropijos nuostolius, vidutini?kai apskai?iuotus visiems ta?kams. Jei kauk? yra dvejetain? (kaip m?s? pavyzdyje) - naudosime **dvejetain?s kry?min?s entropijos nuostolius** (BCE).


In [ ]:
model = SegNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.BCEWithLogitsLoss()

Mokymo ciklas apibr??iamas ?prastu b?du:


In [ ]:
def train(dataloaders, model, loss_fn, optimizer, epochs, device):
    tqdm_iter = tqdm(range(epochs))
    train_dataloader, test_dataloader = dataloaders[0], dataloaders[1]

    for epoch in tqdm_iter:
        model.train()
        train_loss = 0.0
        test_loss = 0.0

        for batch in train_dataloader:
            imgs, labels = batch
            imgs = imgs.to(device)
            labels = labels.to(device)

            preds = model(imgs)
            loss = loss_fn(preds, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        with torch.no_grad():
            for batch in test_dataloader:
                imgs, labels = batch
                imgs = imgs.to(device)
                labels = labels.to(device)

                preds = model(imgs)
                loss = loss_fn(preds, labels)

                test_loss += loss.item()

        train_loss /= len(train_dataloader)
        test_loss /= len(test_dataloader)

        tqdm_dct = {'train loss:': train_loss, 'test loss:': test_loss}
        tqdm_iter.set_postfix(tqdm_dct, refresh=True)
        tqdm_iter.refresh()

In [ ]:
train(dataloaders, model, loss_fn, optimizer, epochs, device)

Nor?dami ?vertinti m?s? model?, tiesiog nubrai?ysime tikslines kaukes ir numatytas kaukes keliems vaizdams:


In [ ]:
model.eval()
predictions = []
image_mask = []
plots = 5
images, masks = test_dataset[0], test_dataset[1]
for i, (img, mask) in enumerate(zip(images, masks)):
    if i == plots:
        break
    img = img.to(device).unsqueeze(0)
    predictions.append((model(img).detach().cpu()[0] > 0.5).float())
    image_mask.append(mask)
plotn(plots, (predictions, image_mask), only_mask=True)

Taip pat yra keletas formali? metrik?, skirt? na?umui ?vertinti, apie kurias galite perskaityti [?ia](https://towardsdatascience.com/metrics-to-evaluate-your-semantic-segmentation-model-6bcb99639aa2). Lengviausia suprasti yra **pikseli? tikslumas** - teisingai klasifikuot? pikseli? procentas.


## U-Net

SegNet architekt?ra yra labai nat?rali, ta?iau ji n?ra pati tiksliausia. I? ties?, pirmiausia taikome piramid?s CNN architekt?r? originaliam vaizdui, kuri suma?ina vaizdo savybi? erdvin? tikslum?. V?liau, kai rekonstruojame vaizd?, negalime tiksliai atkurti pikseli? pozicij?.

Tai veda prie id?jos apie **skip jungtis** tarp konvoliucijos sluoksni? koderyje ir dekoderyje. ?i architekt?ra yra labai da?na semantiniam segmentavimui ir vadinama **U-Net**. Skip jungtys kiekviename konvoliucijos lygyje padeda tinklui neprarasti informacijos apie savybes i? originalaus ?vesties vaizdo tame lygyje.

?ia naudosime gana paprast? CNN architekt?r?, ta?iau U-Net taip pat gali naudoti sud?tingesn? koderyje esant? funkcij? i?gavimo mechanizm?, pavyzd?iui, ResNet-50.

<img src="images/unet.png" width="70%">

> Vaizdas i? straipsnio: Ronneberger, Olaf, Philipp Fischer, ir Thomas Brox. [U-Net: Convolutional networks for biomedical image segmentation.](https://arxiv.org/pdf/1505.04597.pdf)


In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_conv0 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(3,3), padding=1)
        self.act0 = nn.ReLU()
        self.bn0 = nn.BatchNorm2d(16)
        self.pool0 = nn.MaxPool2d(kernel_size=(2,2))

        self.enc_conv1 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3,3), padding=1)
        self.act1 = nn.ReLU()
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=(2,2))

        self.enc_conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3,3), padding=1)
        self.act2 = nn.ReLU()
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 =  nn.MaxPool2d(kernel_size=(2,2))

        self.enc_conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3,3), padding=1)
        self.act3 = nn.ReLU()
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 =  nn.MaxPool2d(kernel_size=(2,2))

        self.bottleneck_conv = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(3,3), padding=1)
        
        self.upsample0 =  nn.UpsamplingBilinear2d(scale_factor=2)
        self.dec_conv0 = nn.Conv2d(in_channels=384, out_channels=128, kernel_size=(3,3), padding=1)
        self.dec_act0 = nn.ReLU()
        self.dec_bn0 = nn.BatchNorm2d(128)

        self.upsample1 =  nn.UpsamplingBilinear2d(scale_factor=2)
        self.dec_conv1 =  nn.Conv2d(in_channels=192, out_channels=64, kernel_size=(3,3), padding=1)
        self.dec_act1 = nn.ReLU()
        self.dec_bn1 = nn.BatchNorm2d(64)

        self.upsample2 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.dec_conv2 = nn.Conv2d(in_channels=96, out_channels=32, kernel_size=(3,3), padding=1)
        self.dec_act2 = nn.ReLU()
        self.dec_bn2 = nn.BatchNorm2d(32)

        self.upsample3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.dec_conv3 = nn.Conv2d(in_channels=48, out_channels=1, kernel_size=(1,1))

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        e0 = self.pool0(self.bn0(self.act0(self.enc_conv0(x))))
        e1 = self.pool1(self.bn1(self.act1(self.enc_conv1(e0))))
        e2 = self.pool2(self.bn2(self.act2(self.enc_conv2(e1))))
        e3 = self.pool3(self.bn3(self.act3(self.enc_conv3(e2))))

        cat0 = self.bn0(self.act0(self.enc_conv0(x)))
        cat1 = self.bn1(self.act1(self.enc_conv1(e0)))
        cat2 = self.bn2(self.act2(self.enc_conv2(e1)))
        cat3 = self.bn3(self.act3(self.enc_conv3(e2)))

        b = self.bottleneck_conv(e3)

        d0 = self.dec_bn0(self.dec_act0(self.dec_conv0(torch.cat((self.upsample0(b), cat3), dim=1))))
        d1 = self.dec_bn1(self.dec_act1(self.dec_conv1(torch.cat((self.upsample1(d0), cat2), dim=1))))
        d2 = self.dec_bn2(self.dec_act2(self.dec_conv2(torch.cat((self.upsample2(d1), cat1), dim=1))))
        d3 = self.sigmoid(self.dec_conv3(torch.cat((self.upsample3(d2), cat0), dim=1)))
        return d3

In [ ]:
model = UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
train(dataloaders, model, loss_fn, optimizer, epochs, device)

In [ ]:
model.eval()
predictions = []
image_mask = []
plots = 5
images, masks = test_dataset[0], test_dataset[1]
for i, (img, mask) in enumerate(zip(images, masks)):
    if i == plots:
        break
    img = img.to(device).unsqueeze(0)
    predictions.append((model(img).detach().cpu()[0] > 0.5).float())
    image_mask.append(mask)
plotn(plots, (predictions, image_mask), only_mask=True)


---

**Atsakomyb?s apribojimas**:  
?is dokumentas buvo i?verstas naudojant AI vertimo paslaug? [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, pra?ome atkreipti d?mes?, kad automatiniai vertimai gali tur?ti klaid? ar netikslum?. Originalus dokumentas jo gimt?ja kalba tur?t? b?ti laikomas autoritetingu ?altiniu. Kritinei informacijai rekomenduojama naudoti profesional? ?mogaus vertim?. Mes neprisiimame atsakomyb?s u? nesusipratimus ar klaidingus interpretavimus, atsiradusius d?l ?io vertimo naudojimo.
